In [ ]:

import pandas as pd

# Load the datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/train.csv'
test_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# Display the first few rows of the datasets
print(train_df.head())
print(test_df.head())


   gender   age  hypertension  ...  HbA1c_level blood_glucose_level  diabetes
0    Male  73.0             0  ...          3.5                  80         0
1    Male  80.0             0  ...          5.7                 145         1
2  Female  38.0             0  ...          4.0                 158         0
3  Female  26.0             0  ...          5.0                 100         0
4  Female  61.0             1  ...          4.5                  85         0

[5 rows x 9 columns]
   gender   age  hypertension  ...  HbA1c_level blood_glucose_level  diabetes
0  Female  13.0             0  ...          5.8                 126         0
1  Female   3.0             0  ...          5.0                 145         0
2    Male  63.0             0  ...          3.5                 200         0
3  Female   2.0             0  ...          6.1                 126         0
4  Female  33.0             0  ...          6.2                 200         1

[5 rows x 9 columns]


In [ ]:

# Step 1: Drop duplicate rows
train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)

# Step 2: Perform one-hot encoding on the "gender" and "smoking_history" columns
train_df_encoded = pd.get_dummies(train_df, columns=['gender', 'smoking_history'])
test_df_encoded = pd.get_dummies(test_df, columns=['gender', 'smoking_history'])

# Display the first few rows of the encoded datasets
print(train_df_encoded.head())
print(test_df_encoded.head())


    age  hypertension  ...  smoking_history_never  smoking_history_not current
0  73.0             0  ...                  False                        False
1  80.0             0  ...                   True                        False
2  38.0             0  ...                  False                        False
3  26.0             0  ...                  False                         True
4  61.0             1  ...                  False                        False

[5 rows x 16 columns]
    age  hypertension  ...  smoking_history_never  smoking_history_not current
0  13.0             0  ...                  False                        False
1   3.0             0  ...                  False                        False
2  63.0             0  ...                  False                        False
3   2.0             0  ...                   True                        False
4  33.0             0  ...                  False                         True

[5 rows x 16 columns]


In [ ]:


import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import numpy as np

# Step 3: Rank the most important features
X_train = train_df_encoded.drop('diabetes', axis=1)
y_train = train_df_encoded['diabetes']

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Feature importances
feature_importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Feature importances:")
print(feature_importances)

# Step 4: Build a Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)

# Step 5: Fit the model on the training set
rf_classifier.fit(X_train, y_train)

# Step 6: Make predictions on the test set
X_test = test_df_encoded.drop('diabetes', axis=1)
y_test = test_df_encoded['diabetes']
y_pred = rf_classifier.predict(X_test)
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]

# Step 7: Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"Area under ROC curve: {roc_auc}")

# Step 8: Show the confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

# Classification report for additional metrics
print("Classification report:")
print(classification_report(y_test, y_pred))



Feature importances:
HbA1c_level                    0.407970
blood_glucose_level            0.312457
bmi                            0.125713
age                            0.105406
hypertension                   0.015265
heart_disease                  0.010795
smoking_history_No Info        0.003989
smoking_history_former         0.003733
smoking_history_never          0.003394
smoking_history_current        0.002612
gender_Male                    0.002222
gender_Female                  0.002198
smoking_history_not current    0.002194
smoking_history_ever           0.002050
gender_Other                   0.000002
dtype: float64
Area under ROC curve: 0.9605355070919468
Confusion matrix:
[[18035    64]
 [  533  1175]]
Classification report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     18099
           1       0.95      0.69      0.80      1708

    accuracy                           0.97     19807
   macro avg       0.96      0.84